In [ ]:
"""
World Cup 2026 predictions v2 — adds altitude & heat-mismatch features.

New features vs v1:
- venue_alt:    altitude (km) of the match city
- alt_shock_h/a: how much higher the venue is than each team's usual
                 home conditions (the away team in La Paz gets a big value)
- heat_h/a:     how much hotter the match climate is than each team's
                 home climate (proxy from latitude + month + solar declination)

For 2026 fixtures, the 16 venues are tagged with real altitude, roof /
climate-control status, and typical June match temperatures.
"""

import numpy as np
import pandas as pd
from math import sin, radians
from collections import defaultdict, deque
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from countryinfo import CountryInfo

# ---------------------------------------------------------------- geography
MANUAL_LATLNG = {
    "England": 52.5, "Wales": 52.3, "Northern Ireland": 54.6, "Kosovo": 42.6,
    "Curaçao": 12.2, "Scotland": 56.5, "Republic of Ireland": 53.3,
    "Ireland": 53.3, "Chinese Taipei": 23.7, "Taiwan": 23.7,
    "China PR": 35.0, "Korea Republic": 36.5, "Korea DPR": 40.0,
    "North Korea": 40.0, "Congo": -1.0, "Gambia": 13.4, "Comoros": -11.7,
    "São Tomé and Príncipe": 0.2, "Timor-Leste": -8.8, "Palestine": 31.9,
    "Micronesia": 6.9, "Vatican City": 41.9, "Zaire": -2.9, "Yugoslavia": 44.0,
    "Czechoslovakia": 49.8, "German DR": 52.0, "Soviet Union": 55.0,
    "Serbia and Montenegro": 44.0, "Saint Martin": 18.1, "Sint Maarten": 18.0,
    "Bonaire": 12.2, "Saint Barthélemy": 17.9, "Guadeloupe": 16.25,
    "Martinique": 14.6, "French Guiana": 4.0, "Réunion": -21.1,
    "Netherlands Antilles": 12.2, "Tahiti": -17.7, "New Caledonia": -21.3,
    "Cook Islands": -21.2, "American Samoa": -14.3, "Samoa": -13.8,
    "Tonga": -21.2, "Fiji": -17.7, "Vanuatu": -16.0, "Solomon Islands": -9.6,
    "Papua New Guinea": -6.5, "Kiribati": 1.4, "Tuvalu": -8.0,
    "Zanzibar": -6.2, "Basque Country": 43.0, "Catalonia": 41.6,
    "Brittany": 48.2, "Corsica": 42.0, "Kernow": 50.4, "Yorkshire": 53.9,
    "Jersey": 49.2, "Guernsey": 49.45, "Isle of Man": 54.2, "Gibraltar": 36.1,
    "Faroe Islands": 62.0, "Greenland": 64.2, "Åland Islands": 60.1,
    "Monaco": 43.7, "Andorra": 42.5, "San Marino": 43.9,
    "Hong Kong": 22.3, "Macau": 22.2, "Tibet": 29.6, "Abkhazia": 43.0,
    "South Ossetia": 42.3, "Northern Cyprus": 35.2, "Crimea": 45.0,
    "Somaliland": 9.5, "Darfur": 13.0, "Western Sahara": 24.0,
    "Saint Kitts and Nevis": 17.3, "Saint Lucia": 13.9,
    "Saint Vincent and the Grenadines": 13.25, "Antigua and Barbuda": 17.05,
    "Trinidad and Tobago": 10.7, "Turks and Caicos Islands": 21.8,
    "Cayman Islands": 19.3, "British Virgin Islands": 18.4,
    "United States Virgin Islands": 18.3, "Anguilla": 18.2,
    "Montserrat": 16.7, "Bermuda": 32.3, "Aruba": 12.5,
    "Eswatini": -26.5, "Swaziland": -26.5, "United Arab Emirates": 24.0,
    "Bosnia and Herzegovina": 44.0, "DR Congo": -2.9,
}
_latlng_cache = {}
def latitude(name):
    if name in _latlng_cache: return _latlng_cache[name]
    lat = MANUAL_LATLNG.get(name)
    if lat is None:
        try:
            ll = CountryInfo(name).latlng()
            lat = ll[0] if ll else None
        except Exception:
            lat = None
    _latlng_cache[name] = lat
    return lat

# Typical altitude (m) where the NATIONAL TEAM usually plays at home.
TEAM_ALT = {
    "Bolivia": 3600, "Ecuador": 2780, "Colombia": 2600, "Mexico": 2240,
    "Ethiopia": 2355, "Eritrea": 2325, "Yemen": 2250, "Bhutan": 2300,
    "Lesotho": 1600, "Kenya": 1795, "Rwanda": 1567, "Burundi": 1820,
    "Zimbabwe": 1490, "South Africa": 1400, "Zambia": 1280, "Uganda": 1190,
    "Afghanistan": 1790, "Nepal": 1350, "Guatemala": 1500, "Armenia": 1000,
    "Iran": 1200, "Kyrgyzstan": 800, "Tajikistan": 800, "Madagascar": 1280,
    "Malawi": 1050, "Eswatini": 1100, "Swaziland": 1100, "Tanzania": 900,
}
# Match-city altitude (m) overrides for notable venues in the dataset.
CITY_ALT = {
    "La Paz": 3640, "El Alto": 4090, "Quito": 2850, "Bogotá": 2640,
    "Bogota": 2640, "Mexico City": 2240, "Toluca": 2660, "Cusco": 3400,
    "Cuzco": 3400, "Sucre": 2810, "Cochabamba": 2560, "Arequipa": 2335,
    "Addis Ababa": 2355, "Addis Abeba": 2355, "Asmara": 2325,
    "Sana'a": 2250, "Sanaa": 2250, "Thimphu": 2320, "Kathmandu": 1400,
    "Quetzaltenango": 2330, "Guatemala City": 1500, "Ciudad de Guatemala": 1500,
    "Johannesburg": 1750, "Pretoria": 1350, "Bloemfontein": 1400,
    "Nairobi": 1795, "Kigali": 1567, "Bujumbura": 780, "Harare": 1490,
    "Lusaka": 1280, "Kampala": 1190, "Antananarivo": 1280, "Maseru": 1600,
    "Tehran": 1200, "Yerevan": 1000, "Kabul": 1790, "Guadalajara": 1550,
    "Zapopan": 1550, "Puebla": 2135, "Pachuca": 2400, "San Luis Potosí": 1860,
    "Aguascalientes": 1880, "León": 1800, "Querétaro": 1820, "Cuenca": 2560,
    "Ambato": 2580, "Ibarra": 2225, "Latacunga": 2760, "Riobamba": 2750,
    "Medellín": 1495, "Manizales": 2160, "Tunja": 2820, "Ipiales": 2900,
    "Denver": 1610, "Calgary": 1045, "Windhoek": 1660, "Gaborone": 1010,
}
def city_altitude(city, country):
    if city in CITY_ALT: return CITY_ALT[city]
    return TEAM_ALT.get(country, 100)  # country default, else ~sea level

# Crude climate proxies. The GBM only needs a monotone-ish signal.
def match_temp(lat, month):
    """Approx daytime temp (C) at latitude `lat` in `month`."""
    if lat is None: return 22.0
    doy = month * 30.4 - 15
    decl = 23.45 * sin(radians(360 / 365 * (284 + doy)))  # solar declination
    return float(np.clip(31 - 0.55 * abs(lat - decl), -5, 38))

def home_climate(lat):
    """Approx warm-season temp a team is accustomed to."""
    if lat is None: return 22.0
    return float(np.clip(29 - 0.30 * abs(lat), 5, 32))

# ---------------------------------------------------------------- load
df = pd.read_csv("results.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)

def k_factor(t):
    t = t.lower()
    if t == "fifa world cup":             return 60
    if "world cup" in t and "qual" in t:  return 50
    if t in ("copa américa", "uefa euro", "african cup of nations",
             "afc asian cup", "gold cup", "uefa nations league"):
        return 50
    if "qual" in t:                       return 40
    if t == "friendly":                   return 20
    return 30

df["k"] = df["tournament"].map(k_factor)
df["importance"] = df["k"] / 60.0
upcoming = df[df.home_score.isna()].copy()
played   = df[df.home_score.notna()].copy()

# coverage check
teams = set(played.home_team) | set(played.away_team)
missing = sorted(t for t in teams if latitude(t) is None)
print(f"Teams without latitude (default climate used): {len(missing)} / {len(teams)}")

# ---------------------------------------------------------------- Elo + features
HOME_ADV = 80
elo  = defaultdict(lambda: 1500.0)
form = defaultdict(lambda: deque(maxlen=10))

def expected(d): return 1.0 / (1.0 + 10 ** (-d / 400.0))
def fstats(f):
    if not f: return (1.2, 1.2, 1.0, 0)
    return (np.mean([x[0] for x in f]), np.mean([x[1] for x in f]),
            np.mean([x[2] for x in f]), len(f))

def climate_feats(home, away, city, country, month):
    alt = city_altitude(city, country)
    alt_shock_h = max(0.0, alt - TEAM_ALT.get(home, 100)) / 1000.0
    alt_shock_a = max(0.0, alt - TEAM_ALT.get(away, 100)) / 1000.0
    mt = match_temp(latitude(country), month)
    heat_h = max(0.0, mt - home_climate(latitude(home)))
    heat_a = max(0.0, mt - home_climate(latitude(away)))
    return alt / 1000.0, alt_shock_h, alt_shock_a, heat_h, heat_a

rows = []
for r in played.itertuples():
    h, a = r.home_team, r.away_team
    eh, ea = elo[h], elo[a]
    adv = 0 if r.neutral else HOME_ADV
    d = eh + adv - ea
    h_gf, h_ga, h_pts, h_n = fstats(form[h])
    a_gf, a_ga, a_pts, a_n = fstats(form[a])
    valt, ash, asa, hh, ha = climate_feats(h, a, r.city, r.country, r.date.month)

    hs, asc = r.home_score, r.away_score
    out = 0 if hs > asc else (1 if hs == asc else 2)
    rows.append((r.date, h, a, eh, ea, d, r.neutral, r.importance,
                 h_gf, h_ga, h_pts, h_n, a_gf, a_ga, a_pts, a_n,
                 valt, ash, asa, hh, ha, out))

    res = 1.0 if out == 0 else (0.5 if out == 1 else 0.0)
    margin = abs(hs - asc)
    g = 1.0 if margin <= 1 else (1.5 if margin == 2 else (11 + margin) / 8.0)
    delta = r.k * g * (res - expected(d))
    elo[h] += delta; elo[a] -= delta
    form[h].append((hs, asc, 3 if out == 0 else (1 if out == 1 else 0)))
    form[a].append((asc, hs, 3 if out == 2 else (1 if out == 1 else 0)))

cols = ["date","home","away","elo_h","elo_a","elo_diff","neutral","importance",
        "h_gf","h_ga","h_pts","h_n","a_gf","a_ga","a_pts","a_n",
        "venue_alt","alt_shock_h","alt_shock_a","heat_h","heat_a","outcome"]
feat = pd.DataFrame(rows, columns=cols)
FEATURES = cols[3:-1]

# ---------------------------------------------------------------- train / validate
pool = feat[feat.date >= "1995-01-01"]
tr, te = pool[pool.date < "2023-01-01"], pool[pool.date >= "2023-01-01"]

model = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06,
                                       max_depth=4, l2_regularization=1.0,
                                       random_state=42)
model.fit(tr[FEATURES], tr["outcome"])
p = model.predict_proba(te[FEATURES])
print(f"\nValidation on {len(te)} matches (2023+):")
print(f"  v2 (climate) - log loss {log_loss(te['outcome'], p):.4f}  "
      f"accuracy {accuracy_score(te['outcome'], p.argmax(1)):.3f}")
print(f"  v1 baseline  - log loss 0.8649  accuracy 0.606  (previous run)")

# Sanity probe: does the model use altitude the right way?
probe = te[te.alt_shock_a > 2.0]
if len(probe):
    pp = model.predict_proba(probe[FEATURES])
    print(f"\nSanity: {len(probe)} test matches with away team >2km above its "
          f"usual altitude\n  actual home-win rate {(probe.outcome==0).mean():.3f}, "
          f"model mean P(home win) {pp[:,0].mean():.3f}")

model.fit(pool[FEATURES], pool["outcome"])

# ---------------------------------------------------------------- 2026 venues
# city -> (altitude m, climate-controlled?, typical June match temp C)
VENUES = {
    "Mexico City":     (2240, False, 24), "Zapopan":        (1550, False, 30),
    "Guadalupe":       (540,  False, 35), "Atlanta":        (300,  True,  22),
    "Arlington":       (180,  True,  22), "Houston":        (15,   True,  22),
    "Kansas City":     (270,  False, 31), "Miami Gardens":  (3,    False, 32),
    "East Rutherford": (3,    False, 28), "Philadelphia":   (12,   False, 29),
    "Foxborough":      (89,   False, 26), "Seattle":        (5,    False, 21),
    "Santa Clara":     (25,   False, 24), "Inglewood":      (30,   True,  23),
    "Vancouver":       (5,    True,  21), "Toronto":        (76,   False, 25),
}
miss = set(upcoming.city) - set(VENUES)
assert not miss, f"unmapped venues: {miss}"

pred_rows = []
for r in upcoming.itertuples():
    h, a = r.home_team, r.away_team
    eh, ea = elo[h], elo[a]
    adv = 0 if r.neutral else HOME_ADV
    alt, roofed, temp = VENUES[r.city]
    h_gf, h_ga, h_pts, h_n = fstats(form[h])
    a_gf, a_ga, a_pts, a_n = fstats(form[a])
    eff_temp = min(temp, 22) if roofed else temp
    heat_h = max(0.0, eff_temp - home_climate(latitude(h)))
    heat_a = max(0.0, eff_temp - home_climate(latitude(a)))
    X = pd.DataFrame([[eh, ea, eh + adv - ea, r.neutral, 1.0,
                       h_gf, h_ga, h_pts, h_n, a_gf, a_ga, a_pts, a_n,
                       alt / 1000.0,
                       max(0.0, alt - TEAM_ALT.get(h, 100)) / 1000.0,
                       max(0.0, alt - TEAM_ALT.get(a, 100)) / 1000.0,
                       heat_h, heat_a]], columns=FEATURES)
    pw, pdr, pl = model.predict_proba(X)[0]
    pick = [h, "Draw", a][int(np.argmax([pw, pdr, pl]))]
    pred_rows.append((r.date.date(), h, a, r.city, alt,
                      "roofed/AC" if roofed else f"open, ~{temp}C",
                      round(pw,3), round(pdr,3), round(pl,3), pick))

out = pd.DataFrame(pred_rows, columns=[
    "date","home","away","city","altitude_m","conditions",
    "P_home_win","P_draw","P_away_win","predicted"])
out.to_csv("wc2026_predictions_v2.csv", index=False)

v1 = pd.read_csv("wc2026_predictions.csv")
cmp = out.merge(v1, on=["home","away"], suffixes=("_v2","_v1"))
cmp["shift"] = (cmp.P_home_win_v2 - cmp.P_home_win_v1).abs() + \
               (cmp.P_away_win_v2 - cmp.P_away_win_v1).abs()
print("\nBiggest prediction shifts vs v1:")
print(cmp.nlargest(8, "shift")[["home","away","city_v2","conditions",
      "P_home_win_v1","P_home_win_v2","P_away_win_v1","P_away_win_v2"]]
      .to_string(index=False))

print("\nMexico's group games:")
print(out[(out.home == "Mexico") | (out.away == "Mexico")].to_string(index=False))

# **Importing Libraries**

In [ ]:
import numpy as np
import pandas as pd
from math import sin, radians
from collections import defaultdict, deque
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from countryinfo import CountryInfo

# **Geographical Features**

In [ ]:
# ---------------------------------------------------------------- geography
MANUAL_LATLNG = {
    "England": 52.5, "Wales": 52.3, "Northern Ireland": 54.6, "Kosovo": 42.6,
    "Curaçao": 12.2, "Scotland": 56.5, "Republic of Ireland": 53.3,
    "Ireland": 53.3, "Chinese Taipei": 23.7, "Taiwan": 23.7,
    "China PR": 35.0, "Korea Republic": 36.5, "Korea DPR": 40.0,
    "North Korea": 40.0, "Congo": -1.0, "Gambia": 13.4, "Comoros": -11.7,
    "São Tomé and Príncipe": 0.2, "Timor-Leste": -8.8, "Palestine": 31.9,
    "Micronesia": 6.9, "Vatican City": 41.9, "Zaire": -2.9, "Yugoslavia": 44.0,
    "Czechoslovakia": 49.8, "German DR": 52.0, "Soviet Union": 55.0,
    "Serbia and Montenegro": 44.0, "Saint Martin": 18.1, "Sint Maarten": 18.0,
    "Bonaire": 12.2, "Saint Barthélemy": 17.9, "Guadeloupe": 16.25,
    "Martinique": 14.6, "French Guiana": 4.0, "Réunion": -21.1,
    "Netherlands Antilles": 12.2, "Tahiti": -17.7, "New Caledonia": -21.3,
    "Cook Islands": -21.2, "American Samoa": -14.3, "Samoa": -13.8,
    "Tonga": -21.2, "Fiji": -17.7, "Vanuatu": -16.0, "Solomon Islands": -9.6,
    "Papua New Guinea": -6.5, "Kiribati": 1.4, "Tuvalu": -8.0,
    "Zanzibar": -6.2, "Basque Country": 43.0, "Catalonia": 41.6,
    "Brittany": 48.2, "Corsica": 42.0, "Kernow": 50.4, "Yorkshire": 53.9,
    "Jersey": 49.2, "Guernsey": 49.45, "Isle of Man": 54.2, "Gibraltar": 36.1,
    "Faroe Islands": 62.0, "Greenland": 64.2, "Åland Islands": 60.1,
    "Monaco": 43.7, "Andorra": 42.5, "San Marino": 43.9,
    "Hong Kong": 22.3, "Macau": 22.2, "Tibet": 29.6, "Abkhazia": 43.0,
    "South Ossetia": 42.3, "Northern Cyprus": 35.2, "Crimea": 45.0,
    "Somaliland": 9.5, "Darfur": 13.0, "Western Sahara": 24.0,
    "Saint Kitts and Nevis": 17.3, "Saint Lucia": 13.9,
    "Saint Vincent and the Grenadines": 13.25, "Antigua and Barbuda": 17.05,
    "Trinidad and Tobago": 10.7, "Turks and Caicos Islands": 21.8,
    "Cayman Islands": 19.3, "British Virgin Islands": 18.4,
    "United States Virgin Islands": 18.3, "Anguilla": 18.2,
    "Montserrat": 16.7, "Bermuda": 32.3, "Aruba": 12.5,
    "Eswatini": -26.5, "Swaziland": -26.5, "United Arab Emirates": 24.0,
    "Bosnia and Herzegovina": 44.0, "DR Congo": -2.9,
}

_latlng_cache = {}

def latitude(name):
    if name in _latlng_cache: return _latlng_cache[name]
    lat = MANUAL_LATLNG.get(name)
    if lat is None:
        try:
            ll = CountryInfo(name).latlng()
            lat = ll[0] if ll else None
        except Exception:
            lat = None
    _latlng_cache[name] = lat
    return lat

# Typical altitude (m) where the NATIONAL TEAM usually plays at home.
TEAM_ALT = {
    "Bolivia": 3600, "Ecuador": 2780, "Colombia": 2600, "Mexico": 2240,
    "Ethiopia": 2355, "Eritrea": 2325, "Yemen": 2250, "Bhutan": 2300,
    "Lesotho": 1600, "Kenya": 1795, "Rwanda": 1567, "Burundi": 1820,
    "Zimbabwe": 1490, "South Africa": 1400, "Zambia": 1280, "Uganda": 1190,
    "Afghanistan": 1790, "Nepal": 1350, "Guatemala": 1500, "Armenia": 1000,
    "Iran": 1200, "Kyrgyzstan": 800, "Tajikistan": 800, "Madagascar": 1280,
    "Malawi": 1050, "Eswatini": 1100, "Swaziland": 1100, "Tanzania": 900,
}
# Match-city altitude (m) overrides for notable venues in the dataset.
CITY_ALT = {
    "La Paz": 3640, "El Alto": 4090, "Quito": 2850, "Bogotá": 2640,
    "Bogota": 2640, "Mexico City": 2240, "Toluca": 2660, "Cusco": 3400,
    "Cuzco": 3400, "Sucre": 2810, "Cochabamba": 2560, "Arequipa": 2335,
    "Addis Ababa": 2355, "Addis Abeba": 2355, "Asmara": 2325,
    "Sana'a": 2250, "Sanaa": 2250, "Thimphu": 2320, "Kathmandu": 1400,
    "Quetzaltenango": 2330, "Guatemala City": 1500, "Ciudad de Guatemala": 1500,
    "Johannesburg": 1750, "Pretoria": 1350, "Bloemfontein": 1400,
    "Nairobi": 1795, "Kigali": 1567, "Bujumbura": 780, "Harare": 1490,
    "Lusaka": 1280, "Kampala": 1190, "Antananarivo": 1280, "Maseru": 1600,
    "Tehran": 1200, "Yerevan": 1000, "Kabul": 1790, "Guadalajara": 1550,
    "Zapopan": 1550, "Puebla": 2135, "Pachuca": 2400, "San Luis Potosí": 1860,
    "Aguascalientes": 1880, "León": 1800, "Querétaro": 1820, "Cuenca": 2560,
    "Ambato": 2580, "Ibarra": 2225, "Latacunga": 2760, "Riobamba": 2750,
    "Medellín": 1495, "Manizales": 2160, "Tunja": 2820, "Ipiales": 2900,
    "Denver": 1610, "Calgary": 1045, "Windhoek": 1660, "Gaborone": 1010,
}

def city_altitude(city, country):
    if city in CITY_ALT: return CITY_ALT[city]
    return TEAM_ALT.get(country, 100)  # country default, else ~sea level

# Crude climate proxies. The GBM only needs a monotone-ish signal.
def match_temp(lat, month):
    """Approx daytime temp (C) at latitude `lat` in `month`."""
    if lat is None: return 22.0
    doy = month * 30.4 - 15
    decl = 23.45 * sin(radians(360 / 365 * (284 + doy)))  # solar declination
    return float(np.clip(31 - 0.55 * abs(lat - decl), -5, 38))

def home_climate(lat):
    """Approx warm-season temp a team is accustomed to."""
    if lat is None: return 22.0
    return float(np.clip(29 - 0.30 * abs(lat), 5, 32))

# **Initialising Variables**

In [ ]:
df = pd.read_csv("results.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)

def k_factor(t):
    t = t.lower()
    if t == "fifa world cup":             return 60
    if "world cup" in t and "qual" in t:  return 50
    if t in ("copa américa", "uefa euro", "african cup of nations",
             "afc asian cup", "gold cup", "uefa nations league"):
        return 50
    if "qual" in t:                       return 40
    if t == "friendly":                   return 20
    return 30

df["k"] = df["tournament"].map(k_factor)
df["importance"] = df["k"] / 60.0
upcoming = df[df.home_score.isna()].copy()
played   = df[df.home_score.notna()].copy()

# coverage check
teams = set(played.home_team) | set(played.away_team)
missing = sorted(t for t in teams if latitude(t) is None)
print(f"Teams without latitude (default climate used): {len(missing)} / {len(teams)}")